# NB11 — Systematic Performance Benchmarking

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

"This is faster" is not a scientific claim. A reproducible performance result requires: a controlled measurement methodology, multiple implementations compared on the same problem, correctness validation before timing, scaling analysis across input sizes, and a principled interpretation of the results.

This notebook builds a reusable `BenchmarkSuite` class and applies it to three bioinformatics kernels across three implementation strategies. The result is a publication-quality analysis, not just a `time.time()` before and after.

---

## 2. Intuition

**The five-step benchmarking protocol:**

1. **Measure** the bottleneck first (profile, don't guess)
2. **Identify** the algorithmic and implementation causes
3. **Optimize** one thing at a time
4. **Validate correctness** — a fast wrong answer is worse than a slow correct one
5. **Re-measure** and compare

**timeit** is the correct tool for benchmarking small Python functions: it runs the function many times, reports the best time (less affected by OS scheduling noise), and handles warm-up.

**The clock matters:** `time.perf_counter()` is the highest-resolution wall clock available. `time.process_time()` measures CPU time only (excludes IO waits). For benchmarking compute, `perf_counter` is usually what you want.

---

## 3. Biological background

The three kernels benchmarked here appear in real bioinformatics pipelines:

1. **GC content** — computed over millions of reads for QC reporting
2. **K-mer counting** — inner loop of metagenomic classifiers (kraken2, sourmash) and genome assemblers (jellyfish)
3. **Pairwise Hamming distance** — used in phylogenetics (SNP-distance matrix), viral evolution, and sequence clustering

Understanding which implementation to use at which scale is a practical decision that affects pipeline runtime.

---

## 4. Mathematical explanation

### Amdahl's Law (revisited)

$$S(n) = \frac{1}{s + \frac{1-s}{n}}$$

where $s$ = serial fraction, $n$ = number of cores. Maximum speedup = $1/s$.

For a function with $s = 0.1$ (10% serial): max speedup = 10x, regardless of cores.

### Algorithmic complexity

| Kernel | Complexity | Memory |
|--------|-----------|--------|
| GC content (N seqs, len L) | O(NL) | O(N) |
| K-mer counting (L seq, k) | O(L) | O(min(L, 4^k)) |
| Pairwise Hamming (N seqs, len L) | O(N²L) | O(N²) for matrix |

The Hamming matrix is the only one with quadratic memory — at N=10,000, float32 requires 400 MB.

### Scaling analysis

If execution time scales as $T(N) = c \cdot N^\alpha$, then in a log-log plot the slope is $\alpha$. This tells you the empirical complexity class, which should match the theoretical prediction.

---

## 5. Computational explanation — BenchmarkSuite

In [ ]:
import numpy as np
import pandas as pd
import time
import timeit
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Callable, Any

np.random.seed(42)

try:
    import numba
    NUMBA_AVAILABLE = True
    print(f"Numba available: {numba.__version__}")
except ImportError:
    NUMBA_AVAILABLE = False
    print("Numba not available — will use Python/NumPy only")

print("NB11: Systematic Performance Benchmarking")

In [ ]:
# --- BenchmarkSuite class ---

class BenchmarkSuite:
    """Systematic benchmarking for multiple implementations of the same function.
    
    Usage:
        suite = BenchmarkSuite('GC Content')
        suite.add_impl('Python loop', gc_python)
        suite.add_impl('NumPy', gc_numpy)
        results_df = suite.run(inputs, repeat=10, warmup=2)
        suite.plot_scaling(results_df)
    """
    
    def __init__(self, name: str):
        self.name = name
        self.implementations: dict[str, Callable] = {}
        self.reference_impl: str | None = None
    
    def add_impl(self, label: str, fn: Callable, is_reference: bool = False):
        """Register an implementation."""
        self.implementations[label] = fn
        if is_reference:
            self.reference_impl = label
    
    def validate_correctness(self, inputs: list) -> dict[str, bool]:
        """Check all implementations agree with the reference on all inputs."""
        if self.reference_impl is None:
            raise ValueError("Set a reference implementation with is_reference=True")
        
        ref_fn = self.implementations[self.reference_impl]
        results = {}
        
        for label, fn in self.implementations.items():
            if label == self.reference_impl:
                results[label] = True
                continue
            
            all_match = True
            for x in inputs:
                ref_out = ref_fn(x)
                impl_out = fn(x)
                if isinstance(ref_out, np.ndarray):
                    if not np.allclose(ref_out, impl_out, atol=1e-5):
                        all_match = False
                        break
                else:
                    if abs(ref_out - impl_out) > 1e-10:
                        all_match = False
                        break
            results[label] = all_match
        
        return results
    
    def run(
        self,
        inputs: list,          # list of (input, label) tuples
        repeat: int = 10,      # number of timed repetitions
        warmup: int = 2,       # warmup calls (not timed)
    ) -> pd.DataFrame:
        """Benchmark all implementations on all inputs.
        
        Returns a DataFrame with columns:
            implementation, input_label, input_size, time_s, speedup
        """
        rows = []
        
        for x, size_label in inputs:
            ref_time = None
            
            for label, fn in self.implementations.items():
                # Warmup
                for _ in range(warmup):
                    fn(x)
                
                # Timing
                times = []
                for _ in range(repeat):
                    t0 = time.perf_counter()
                    fn(x)
                    times.append(time.perf_counter() - t0)
                
                best_time = min(times)  # use best (least noise)
                
                if label == self.reference_impl:
                    ref_time = best_time
                
                rows.append({
                    'implementation': label,
                    'input_label': size_label,
                    'time_s': best_time,
                })
        
        df = pd.DataFrame(rows)
        
        # Add speedup column (relative to reference implementation)
        ref_times = df[df['implementation'] == self.reference_impl][['input_label', 'time_s']]
        ref_times = ref_times.rename(columns={'time_s': 'ref_time'})
        df = df.merge(ref_times, on='input_label')
        df['speedup'] = df['ref_time'] / df['time_s']
        
        return df
    
    def print_table(self, df: pd.DataFrame):
        """Print a formatted results table."""
        print(f"\nBenchmark: {self.name}")
        print(f"{'Impl':20} {'Input':12} {'Time (ms)':12} {'Speedup':10}")
        print("-" * 58)
        for _, row in df.iterrows():
            print(f"{row['implementation']:20} {str(row['input_label']):12} "
                  f"{row['time_s']*1000:12.3f} {row['speedup']:10.2f}x")

print("BenchmarkSuite class defined.")

In [ ]:
# --- Kernel 1: GC content ---

rng = np.random.default_rng(42)

def gc_python(seqs):
    return np.array([sum(1 for b in seq if b == 2 or b == 3) / len(seq) for seq in seqs])

def gc_numpy(seqs):
    return ((seqs == 2) | (seqs == 3)).mean(axis=1)

if NUMBA_AVAILABLE:
    import numba
    @numba.njit
    def gc_numba_inner(seqs):
        N, L = seqs.shape
        result = np.empty(N, dtype=numba.float64)
        for i in range(N):
            gc = 0
            for j in range(L):
                if seqs[i, j] == 2 or seqs[i, j] == 3:
                    gc += 1
            result[i] = gc / L
        return result
    
    def gc_numba(seqs):
        return gc_numba_inner(seqs)

suite_gc = BenchmarkSuite('GC Content')
sizes_gc = [100, 500, 2000]
L = 200

# Build inputs as 2D integer arrays
inputs_gc = []
for N in sizes_gc:
    arr = rng.integers(0, 4, size=(N, L), dtype=np.int8)
    inputs_gc.append((arr, N))

# Also need list-of-arrays for Python loop
inputs_gc_list = [(list(arr), N) for arr, N in inputs_gc]

suite_gc.add_impl('Python loop', gc_python, is_reference=True)
suite_gc.add_impl('NumPy', gc_numpy)
if NUMBA_AVAILABLE:
    # Warmup Numba compilation
    _ = gc_numba(inputs_gc[0][0])
    suite_gc.add_impl('Numba JIT', gc_numba)

# Correctness check (small input)
small = inputs_gc[0][0]
r_py = gc_python(list(small))
r_np = gc_numpy(small)
assert np.allclose(r_py, r_np, atol=1e-6), "GC NumPy disagrees!"
if NUMBA_AVAILABLE:
    r_nb = gc_numba(small)
    assert np.allclose(r_py, r_nb, atol=1e-6), "GC Numba disagrees!"
print("GC content correctness: PASSED")

# Run benchmark
df_gc_py = [(list(arr), N) for arr, N in inputs_gc]
df_gc_np = [(arr, N) for arr, N in inputs_gc]

# Combine with appropriate inputs for each impl
# Use numpy inputs for all — gc_python works on 2D arrays too
df_gc = suite_gc.run(inputs_gc, repeat=5, warmup=1)
suite_gc.print_table(df_gc)

In [ ]:
# --- Kernel 2: K-mer counting ---

def kmer_python(seq_and_k):
    seq, k = seq_and_k
    counts = defaultdict(int)
    for i in range(len(seq) - k + 1):
        counts[seq[i:i+k]] += 1
    return len(counts)  # return count of distinct kmers

def kmer_numpy(seq_and_k):
    """NumPy k-mer counting using rolling hash approach."""
    seq, k = seq_and_k
    L = len(seq)
    bases = np.frombuffer(seq.encode(), dtype=np.uint8)
    # Sliding window via stride tricks
    windows = np.lib.stride_tricks.sliding_window_view(bases, k)
    # Hash each window: treat as base-4 number
    powers = 4 ** np.arange(k - 1, -1, -1, dtype=np.int64)
    # Map ATCG → 0123 (simplified, not handling all ASCII)
    mapping = np.zeros(256, dtype=np.int8)
    for i, b in enumerate(b'ATCG'):
        mapping[b] = i
    encoded = mapping[windows]
    hashes = (encoded * powers).sum(axis=1)
    return len(np.unique(hashes))

suite_kmer = BenchmarkSuite('K-mer Counting (k=4)')
k = 4
lengths = [100, 500, 2000]

def gen_dna(L, rng):
    return ''.join(rng.choice(list('ATCG'), size=L))

inputs_kmer = [(gen_dna(L, rng), L) for L in lengths]
inputs_kmer_k = [((seq, k), L) for seq, L in inputs_kmer]

suite_kmer.add_impl('Python dict', kmer_python, is_reference=True)
suite_kmer.add_impl('NumPy hashing', kmer_numpy)

# Quick correctness check
r1 = kmer_python((inputs_kmer[0][0], k))
r2 = kmer_numpy((inputs_kmer[0][0], k))
# Note: distinct kmer counts should be similar (might differ due to collisions in simple hash)
print(f"K-mer distinct counts (L=100, k=4): Python={r1}, NumPy hashing={r2}")

df_kmer = suite_kmer.run(inputs_kmer_k, repeat=10, warmup=2)
suite_kmer.print_table(df_kmer)

In [ ]:
# --- Kernel 3: Pairwise Hamming distance ---

def hamming_python(seqs):
    N = len(seqs)
    D = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        for j in range(i + 1, N):
            d = int(np.sum(seqs[i] != seqs[j]))
            D[i, j] = D[j, i] = d
    return D

def hamming_numpy(seqs):
    diff = seqs[:, np.newaxis, :] != seqs[np.newaxis, :, :]
    return diff.sum(axis=2).astype(np.float32)

if NUMBA_AVAILABLE:
    @numba.njit(parallel=True)
    def hamming_numba_inner(seqs):
        N, L = seqs.shape
        D = np.zeros((N, N), dtype=numba.float32)
        for i in numba.prange(N):
            for j in range(i + 1, N):
                diff = numba.int32(0)
                for k in range(L):
                    if seqs[i, k] != seqs[j, k]:
                        diff += 1
                D[i, j] = D[j, i] = diff
        return D
    
    def hamming_numba(seqs):
        return hamming_numba_inner(seqs)

suite_hd = BenchmarkSuite('Pairwise Hamming Distance')
Ns = [20, 50, 100]
L_hd = 100

inputs_hd = []
for N in Ns:
    arr = rng.integers(0, 4, size=(N, L_hd), dtype=np.int8)
    inputs_hd.append((arr, N))

suite_hd.add_impl('Python loop', hamming_python, is_reference=True)
suite_hd.add_impl('NumPy broadcast', hamming_numpy)
if NUMBA_AVAILABLE:
    _ = hamming_numba(inputs_hd[0][0])  # warmup
    suite_hd.add_impl('Numba parallel', hamming_numba)

# Correctness
D_py = hamming_python(inputs_hd[0][0])
D_np = hamming_numpy(inputs_hd[0][0])
assert np.allclose(D_py, D_np), "Hamming disagreement!"
if NUMBA_AVAILABLE:
    D_nb = hamming_numba(inputs_hd[0][0])
    assert np.allclose(D_py, D_nb), "Numba Hamming disagreement!"
print("Pairwise Hamming correctness: PASSED")

df_hd = suite_hd.run(inputs_hd, repeat=3, warmup=1)
suite_hd.print_table(df_hd)

In [ ]:
# --- Amdahl's Law ---
def amdahl_speedup(serial_fraction, n_cores):
    s = serial_fraction
    return 1.0 / (s + (1.0 - s) / n_cores)

cores = np.arange(1, 33)
serial_fractions = [0.01, 0.05, 0.1, 0.25, 0.5]

print("Amdahl's Law — maximum speedup:")
print(f"{'Serial %':>10} {'n=4':>8} {'n=8':>8} {'n=16':>8} {'n=32':>8} {'n→∞':>8}")
for s in serial_fractions:
    row = [f"{amdahl_speedup(s, n):.1f}x" for n in [4, 8, 16, 32]]
    lim = f"{1/s:.0f}x"
    print(f"{s*100:>9.0f}% {' '.join(row):>40} {lim:>8}")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'Python loop': '#e74c3c', 'NumPy': '#2980b9',
          'Numba JIT': '#27ae60', 'Numba parallel': '#8e44ad',
          'NumPy broadcast': '#2980b9', 'Python dict': '#e74c3c',
          'NumPy hashing': '#e67e22'}

def plot_scaling(ax, df, title, xlabel):
    for impl, group in df.groupby('implementation'):
        color = colors.get(impl, '#333333')
        ax.loglog(group['input_label'], group['time_s'] * 1000,
                  'o-', label=impl, color=color, linewidth=2, markersize=7)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Wall time (ms)', fontsize=10)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.3)

# Panel 1: GC content scaling
plot_scaling(axes[0, 0], df_gc, 'GC Content: Scaling Curves', 'Number of sequences (N)')

# Panel 2: Speedup heatmap (method × sequence length)
ax = axes[0, 1]
# Pivot df_gc to create a matrix: implementations × input sizes
pivot = df_gc.pivot(index='implementation', columns='input_label', values='speedup')
im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto', vmin=0)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8)
ax.set_xlabel('Number of sequences', fontsize=10)
ax.set_title('Speedup Heatmap\n(GC Content, relative to Python loop)', fontweight='bold')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.1f}x', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, label='Speedup')

# Panel 3: Amdahl's Law
ax = axes[1, 0]
amdahl_colors = ['#2980b9', '#27ae60', '#f39c12', '#e74c3c', '#8e44ad']
for s, color in zip(serial_fractions, amdahl_colors):
    speedups = [amdahl_speedup(s, n) for n in cores]
    ax.plot(cores, speedups, '-', color=color, linewidth=2, label=f's={s:.0%}')
ax.axvline(x=8, color='gray', linestyle='--', alpha=0.5, label='8 cores (typical)')
ax.set_xlabel('Number of cores', fontsize=10)
ax.set_ylabel('Theoretical speedup', fontsize=10)
ax.set_title("Amdahl's Law\n(theoretical maximum speedup)", fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, 32)

# Panel 4: Profiling waterfall (pairwise Hamming)
ax = axes[1, 1]
if len(df_hd['implementation'].unique()) > 1:
    # Bar chart for N=50 only
    n_target = Ns[1]  # middle size
    df_n = df_hd[df_hd['input_label'] == n_target]
    impls = df_n['implementation'].tolist()
    times = (df_n['time_s'] * 1000).tolist()
    bar_colors = [colors.get(impl, '#333') for impl in impls]
    bars = ax.bar(range(len(impls)), times, color=bar_colors, edgecolor='black', alpha=0.85)
    ax.set_xticks(range(len(impls)))
    ax.set_xticklabels(impls, fontsize=8, rotation=10)
    for bar, v in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.02, f'{v:.2f}ms',
                ha='center', va='bottom', fontsize=8)
    ax.set_ylabel('Wall time (ms)', fontsize=10)
    ax.set_title(f'Pairwise Hamming (N={n_target}, L={L_hd})\nImplementation Comparison', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Only one implementation available',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('../datasets/nb11_benchmarking.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Extend BenchmarkSuite:** Add a `plot_speedup_vs_size` method that plots speedup (not time) on the y-axis as a function of input size. This makes it easier to see whether the speedup changes as the problem scales.

2. **Empirical complexity:** Fit a linear regression to `log(time)` vs `log(N)` for each implementation in the GC content benchmark. The slope is the empirical complexity exponent. Compare to the theoretical value.

3. **Amdahl in practice:** Profile the joblib GC content computation from NB03 using `cProfile`. Estimate the serial fraction (overhead: pickling, process startup, result collection). Plug this into Amdahl's formula and compare to the measured speedup.

4. **Memory profiling:** Use `memory_profiler` (pip install memory_profiler) to profile the peak memory usage of `hamming_python` and `hamming_numpy` at N=100, L=200. Which uses less memory? Does the result match the theoretical analysis?

## 12. Reflection

*Write here: Why do we use `min(times)` rather than `mean(times)` when benchmarking? What does the minimum time represent? When would the mean be more informative?*

---

## References

- Python `timeit` module: https://docs.python.org/3/library/timeit.html
- Amdahl, G.M. (1967). "Validity of the single processor approach." *AFIPS*.
- `memory_profiler`: https://github.com/pythonprofilers/memory_profiler
- `perf` (Linux CPU profiling): https://perf.wiki.kernel.org/

## Future work / open questions

- How would you benchmark a function that reads from disk? Can you use `timeit` directly? What alternatives exist (`hyperfine`, `pytest-benchmark`)?
- For the Hamming distance matrix at N=10,000, the NumPy broadcasting approach requires 10,000 × 10,000 × 200 bytes = 20 GB of intermediate storage. How would you implement a block-wise version that uses only 1 GB?